In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pzest.config import load_config
from pzest.dataset.splits import make_splits
from pzest.dataset.dataset import get_dataloader
from pzest.training.sampling import compute_sample_weights

In [ ]:
config = load_config("../config/default.yaml")

In [ ]:
desi_hsc_filepath = "../data/raw/DESI_HSC_64x64_v2.hdf5"

Check file structure

In [ ]:
with h5py.File("../data/raw/DESI_HSC_64x64_v2.hdf5", "r") as f:
    print("Top level keys:", list(f.keys()))
    for key in f.keys():
        obj = f[key]
        if hasattr(obj, "keys"):
            print(f"\n{key}/ (group)")
            for subkey in obj.keys():
                subobj = obj[subkey]
                if hasattr(subobj, "shape"):
                    print(f"{subkey}: shape={subobj.shape}, dtype={subobj.dtype}")
                else:
                    print(f"{subkey}/ (subgroup)")
        else:
            print(f"\n{key}: shape={obj.shape}, dtype={obj.dtype}")

Check redshift distribution

In [ ]:
with h5py.File(desi_hsc_filepath, "r") as f:
    redshifts = f["DESI_fibermap"]["DESI_redshift"][:]
    images = f["image"][:5]
    morphtypes = f["DESI_fibermap"]["MORPHTYPE"][:]

# Redshift distribution
plt.figure(figsize=(10,4))
plt.hist(redshifts, bins=100, edgecolor="none")
plt.xlabel("Redshift (z)")
plt.ylabel("Count")
plt.title("Redshift distribution")
plt.show()

# Morphology types
unique, counts = np.unique(morphtypes, return_counts=True)
for m, c in zip(unique, counts):
    print(f"{m.decode()}: {c} ({100*c/len(morphtypes):.1f}%) ")

# Sample images - show first galaxy across all 5 bands
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
band_names = ["g", "r", "i", "z", "y"]
for i, (ax, band) in enumerate(zip(axes, band_names)):
    ax.imshow(images[0, i], origin="lower", cmap="viridis")
    ax.set_title(f"{band} band")
    ax.axis("off")
plt.suptitle("Galaxy 0 - all 5 bands")
plt.tight_layout()
plt.show()

Check morphology

In [ ]:
with h5py.File(desi_hsc_filepath, "r") as f:
    redshifts = f["DESI_fibermap"]["DESI_redshift"][:]
    morphtypes = f["DESI_fibermap"]["MORPHTYPE"][:]

print(f"Total galaxies: {len(redshifts)}")
print(f"z < 0.5: {(redshifts < 0.5).sum()} ({100*(redshifts < 0.5).mean():.1f}%)")
print(f"z < 1.0: {(redshifts < 1.0).sum()} ({100*(redshifts < 1.0).mean():.1f}%)")
print(f"z < 2.0: {(redshifts < 2.0).sum()} ({100*(redshifts < 2.0).mean():.1f}%)")
print(f"z > 2.0: {(redshifts > 2.0).sum()} ({100*(redshifts > 2.0).mean():.1f}%)")

# Check PSF objects redshift distribution
psf_mask = np.array([m.decode().strip() == "PSF" for m in morphtypes])
print(f"\nPSF objects redshift stats:")
print(f"median z: {np.median(redshifts[psf_mask]):.3f}")
print(f"mean z: {np.mean(redshifts[psf_mask]):.3f}")
print(f"PSF objects at z < 0.5: {(redshifts[psf_mask] < 0.5).sum()}")

Check WISE flux quality and spectrum SNR

In [ ]:
with h5py.File(desi_hsc_filepath, "r") as f:
    redshifts = f["DESI_fibermap"]["DESI_redshift"][:]
    flux_w1 = f["DESI_fibermap"]["FLUX_W1"][:]
    flux_w2 = f["DESI_fibermap"]["FLUX_W2"][:]
    flux_r = f["DESI_fibermap"]["FLUX_R"][:]
    ivar = f["spectrum"]["ivar"][:10]
    flux = f["spectrum"]["flux"][:10]
    wave = f["spectrum"]["wave"][:10]

# Check WISE flux quality
print("FLUX_W1 stats:")
print(f"non-zero: {(flux_w1 != 0).sum()} ({100*(flux_w1 != 0).mean():.1f}%)")
print(f"positive: {(flux_w1 > 0).sum()} ({100*(flux_w1 > 0).mean():.1f}%)")

# Check spectrum SNR for a few galaxies
snr = flux * np.sqrt(ivar)
median_snr = np.nanmedian(snr, axis=1)
print(f"\nMedian SNR for first 10 spectra: {median_snr.round(2)}")

# Plot a sample spectrum
plt.figure(figsize=(12, 4))
plt.plot(wave[0], flux[0], lw=0.5, alpha=0.8)
plt.xlabel("Wavelength (Å)")
plt.ylabel("Flux")
plt.title(f"Sample spectrum (z={redshifts[0]:.3f})")
plt.show()

Check low-redshift spectrum

In [ ]:
# Find a low redshift galaxy
with h5py.File(desi_hsc_filepath, "r") as f:
    redshifts = f["DESI_fibermap"]["DESI_redshift"][:]
    morphtypes = f["DESI_fibermap"]["MORPHTYPE"][:]

    low_z_mask = (redshifts < 0.2)
    non_psf_mask = np.array([m.decode().strip() != "PSF" for m in morphtypes])
    good_mask = low_z_mask & non_psf_mask
    idx = np.where(good_mask)[0][0]

    flux = f["spectrum"]["flux"][:idx]
    wave = f["spectrum"]["wave"][:idx]
    image = f["image"][idx]
    z = redshifts[idx]
    morph = morphtypes[idx].decode().strip()

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
band_names = ["g", "r", "i", "z", "y"]
for i, (ax, band) in enumerate(zip(axes[:5], band_names)):
    ax.imshow(image[i], origin="lower", cmap="viridis")
    ax.set_title(f"{band} band")
    ax.axis("off")

axes[5].plot(wave, flux, lw=0.5)
axes[5].set_xlabel("Wavelength (Å)")
axes[5].set_ylabel("Flux")
axes[5].set_title(f"Spectrum (z={z:.3f}, {morph})")
plt.tight_layout()
plt.show()

Check SNR distribution

In [ ]:
with h5py.File(desi_hsc_filepath, "r") as f:
    # Compute median SNR per spectrum across all galaxies
    # Read in chunks to avoid memory issues
    n = 134533
    chunk = 1000
    median_snr = np.zeros(n)

    for i in range(0, n, chunk):
        flux_chunk = f["spectrum"]["flux"][i:i+chunk]
        ivar_chunk = f["spectrum"]["ivar"][i:i+chunk]
        snr_chunk = flux_chunk * np.sqrt(np.clip(ivar_chunk, 0, None))
        median_snr[i:i+chunk] = np.nanmedian(snr_chunk, axis=1)

print(f"SNR distribution across all {n} galaxies:")
print(f"median: {np.median(median_snr):.2f}")
print(f"mean: {np.mean(median_snr):.2f}")
print(f"SNR > 5: {(median_snr > 5).sum()} ({100*(median_snr > 5).mean():.1f}%)")
print(f"SNR > 10: {(median_snr > 10).sum()} ({100*(median_snr > 10).mean():.1f}%)")
print(f"SNR > 2: {(median_snr > 2).sum()} ({100*(median_snr > 2).mean():.1f}%)")

plt.figure(figsize=(10, 4))
plt.hist(median_snr, bins=100, range=(0, 30))
plt.xlabel("Median SNR per spectrum")
plt.ylabel("Count")
plt.title("Spectral SNR distribution")
plt.axvline(5, color="red", linestyle="dashed", label="SNR=5")
plt.axvline(10, color="orange", linestyle="dashed", label="SNR=10")
plt.legend()
plt.show()

Check quality of redshift, magnitude and image

In [ ]:
with h5py.File(desi_hsc_filepath, "r") as f:
    redshifts = f["DESI_fibermap"]["DESI_redshift"][:]
    redshift_err = f["DESI_fibermap"]["DESI_redshift_err"][:]
    morphtypes = f["DESI_fibermap"]["MORPHTYPE"][:]

    # HSC magnitudes
    g_mag = f["HSC_metadata"]["g_cmodel_mag"][:]
    r_mag = f["HSC_metadata"]["r_cmodel_mag"][:]
    i_mag = f["HSC_metadata"]["i_cmodel_mag"][:]
    z_mag = f["HSC_metadata"]["z_cmodel_mag"][:]
    y_mag = f["HSC_metadata"]["y_cmodel_mag"][:]
    specz_flag = f["HSC_metadata"]["specz_flag_homogeneous"][:]

    # Images - check a sample for NaN
    images_sample = f["image"][:1000]

In [ ]:
# Redshift quality
print("=== Redshift Quality ===")
print(f"NaN redshifts: {np.isnan(redshifts).sum()}")
print(f"Negative redshifts: {(redshifts < 0).sum()}")
print(f"redshift_err NaN: {np.isnan(redshift_err).sum()}")
print(f"redshift_err > 0.01: {(redshift_err > 0.01).sum()} ({100*(redshift_err > 0.01).mean():.1f}%)")
print(f"redshift_err > 0.1: {(redshift_err > 0.1).sum()} ({100*(redshift_err > 0.1).mean():.1f}%)")

In [ ]:
# Magnitude quality
print("\n=== Magnitude Quality ===")
for name, mag in zip(["g", "r", "i", "z", "y"], [g_mag, r_mag, i_mag, z_mag, y_mag]):
    nan_count = np.isnan(mag).sum()
    inf_count = np.isinf(mag).sum()
    print(f"{name}_cmodel_mag: NaN={nan_count}, Inf={inf_count}, min={np.nanmin(mag):.2f}, max={np.nanmax(mag):.2f}")

In [ ]:
# specz_flag
print("\n=== specz_flag_homogeneous ===")
print(f"True: {specz_flag.sum()} ({100*specz_flag.mean():.1f}%)")
print(f"False: {(~specz_flag).sum()} ({100*(~specz_flag).mean():.1f}%)")

In [ ]:
# Image quality
print("\n=== Image Quality (first 1000) ===")
print(f"NaN pixels: {np.isnan(images_sample).sum()}")
print(f"Inf pixels: {np.isinf(images_sample).sum()}")
print(f"Min value: {images_sample.min():.4f}")
print(f"Max value: {images_sample.max():.4f}")
print(f"Mean value: {images_sample.mean():.4f}")
print(f"Std value: {images_sample.std():.4f}")

Check per-channel stats to inform normalisation

In [ ]:
with h5py.File(desi_hsc_filepath, "r") as f:
    n = 134533
    chunk = 500
    band_names = ["g", "r", "i", "z", "y"]

    # Accumulate statistics
    channel_sum = np.zeros(5)
    channel_sum_sq = np.zeros(5)
    total_pixels = 0

    for i in range(0, n, chunk):
        batch = f["image"][i:i+chunk]       # (chunk, 5, 64, 64)
        channel_sum += batch.sum(axis=(0, 2, 3))
        channel_sum_sq += (batch ** 2).sum(axis=(0, 2, 3))
        total_pixels += batch.shape[0] * 64 * 64

channel_mean = channel_sum / total_pixels
channel_std = np.sqrt(channel_sum_sq / total_pixels - channel_mean ** 2)

print("Per-channel statistics across full dataset:")
for band, mean, std in zip(band_names, channel_mean, channel_std):
    print(f"{band}: mean={mean:.4f}, std={std:.4f}")

In [ ]:
with h5py.File(desi_hsc_filepath, "r") as f:
    images_sample = f["image"][:5000]

channel_mean = np.array([0.2851, 0.6045, 0.9089, 1.1676, 1.3929])
channel_std  = np.array([1.1039, 2.1952, 3.5377, 4.3867, 5.0706])

# Check what fraction of pixels are beyond 5 and 10 sigma
for i, band in enumerate(["g", "r", "i", "z", "y"]):
    band_pixels = images_sample[:, i, :, :]
    normalised = (band_pixels - channel_mean[i]) / channel_std[i]
    beyond_5 = (np.abs(normalised) > 5).mean()
    beyond_10 = (np.abs(normalised) > 10).mean()
    print(f"{band}: >5σ = {100*beyond_5:.2f}%, >10σ = {100*beyond_10:.2f}%")

In [ ]:
with h5py.File(desi_hsc_filepath, "r") as f:
    images_sample = f["image"][:1000]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
band_names = ["g", "r", "i", "z", "y"]

for i, (band, ax_raw, ax_stretch) in enumerate(
    zip(band_names, axes[0], axes[1])):
    pixels = images_sample[:, i, :, :].flatten()

    # Raw distribution - clip for visibility
    ax_raw.hist(pixels, bins=100, range=(-2, 20))
    ax_raw.set_title(f"{band} raw")
    ax_raw.set_xlabel("Flux")

    # arcsinh stretched
    stretched = np.arcsinh(pixels / 0.1)
    ax_stretch.hist(stretched, bins=100)
    ax_stretch.set_title(f"{band} arcsinh(x/0.1)")
    ax_stretch.set_xlabel("arcsinh flux")

plt.tight_layout()
plt.show()

In [ ]:
with h5py.File(desi_hsc_filepath, "r") as f:
    images_sample = f["image"][:1000]

band_names = ["g", "r", "i", "z", "y"]
print("Background noise estimates per band:")
for i, band in enumerate(band_names):
    pixels = images_sample[:, i, :, :].flatten()
    neg_pixels = pixels[pixels < 0]
    noise = np.std(neg_pixels) if len(neg_pixels) > 0 else np.percentile(np.abs(pixels), 16)
    print(f"{band}: noise std = {noise:.4f}")

In [ ]:
with h5py.File(desi_hsc_filepath, "r") as f:
    images_sample = f["image"][:1000]

noise_scales = np.array([0.0212, 0.0352, 0.0369, 0.0688, 0.1367])
band_names = ["g", "r", "i", "z", "y"]
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, (band, ax_hist, ax_img) in enumerate(
    zip(band_names, axes[0], axes[1])):
    pixels = images_sample[:, i, :, :].flatten()
    stretched = np.arcsinh(pixels / noise_scales[i])

    ax_hist.hist(stretched, bins=100, range=(-5, 15))
    ax_hist.set_title(f"{band} arcsinh(x/{noise_scales[i]:.4f})")
    ax_hist.set_xlabel("arcsinh flux")

    # Show a sample galaxy image after stretching
    img_stretched = np.arcsinh(images_sample[10, i] / noise_scales[i])
    ax_img.imshow(img_stretched, origin="lower", cmap="viridis")
    ax_img.set_title(f"{band} band (galaxy 10)")
    ax_img.axis("off")

plt.tight_layout()
plt.show()

Re-compute channel mean and std

In [ ]:
band_names = ["g", "r", "i", "z", "y"]

with h5py.File(desi_hsc_filepath, "r") as f:
    n = 134533
    chunk = 500
    arcsinh_scales = np.array([0.0212, 0.0352, 0.0369, 0.0688, 0.1367])

    channel_sum = np.zeros(5)
    channel_sum_sq = np.zeros(5)
    total_pixels = 0

    for i in range(0, n, chunk):
        batch = f["image"][i:i+chunk].astype(np.float32)
        # Apply arcsinh stretch
        for c in range(5):
            batch[:, c] = np.arcsinh(batch[:, c] / arcsinh_scales[c])
        channel_sum += batch.sum(axis=(0, 2, 3))
        channel_sum_sq += (batch ** 2).sum(axis=(0, 2, 3))
        total_pixels += batch.shape[0] * 64 * 64

channel_mean = channel_sum / total_pixels
channel_std = np.sqrt(channel_sum_sq / total_pixels - channel_mean ** 2)

print("Per-channel statistics after arcsinh stretch:")
for band, mean, std in zip(band_names, channel_mean, channel_std):
    print(f"{band}: mean={mean:.4f}, std={std:.4f}")

Check weighted sampling is working as expected

In [ ]:
with h5py.File(config.paths.processed_hdf5_file, "r") as f:
    num_samples = f["images"].shape[0]

train_indices, _, _ = make_splits(config, num_samples)

bin_edges = np.linspace(
    config.model.redshift_min,
    config.model.redshift_max,
    config.model.num_bins + 1,
)

with h5py.File(config.paths.processed_hdf5_file, "r") as f:
    train_redshifts = f["redshift"][np.sort(train_indices)]

# --- Without weighting ---
loader_unweighted = get_dataloader(
    hdf5_path=config.paths.processed_hdf5_file,
    indices=train_indices,
    bin_edges=bin_edges,
    config=config,
    shuffle=True,
)

# --- With weighting ---
sample_weights = compute_sample_weights(train_redshifts, bin_edges)
loader_weighted = get_dataloader(
    hdf5_path=config.paths.processed_hdf5_file,
    indices=train_indices,
    bin_edges=bin_edges,
    config=config,
    shuffle=True,
    sample_weights=sample_weights,
)

# Collect labels from first 10 batches of each
def collect_redshifts(loader, n_batches=10):
    redshifts = []
    bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    for i, batch in enumerate(loader):
        if i >= n_batches:
            break
        labels = batch["label"].numpy()
        redshifts.extend(bin_centres[labels])
    return np.array(redshifts)

z_unweighted = collect_redshifts(loader_unweighted)
z_weighted = collect_redshifts(loader_weighted)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(z_unweighted, bins=50, edgecolor="none")
ax1.set_xlabel("Redshift")
ax1.set_ylabel("Count")
ax1.set_title("Unweighted sampling")

ax2.hist(z_weighted, bins=50, edgecolor="none")
ax2.set_xlabel("Redshift")
ax2.set_ylabel("Count")
ax2.set_title("Weighted sampling")

plt.tight_layout()
plt.show()